In [1]:
print("Hello Conor")

Hello Conor


In [9]:
#need to import packages

try:
    import pyarrow
except ModuleNotFoundError:
    ! pip install pyarrow
    import pyarrow





import numpy as np
import pandas as pd
from urllib.request import urlopen
from pathlib import Path
import gzip


In [ ]:
#Now we read in the files, set it up to be reproducible and changing paths is easy
city = 'London'
ymd = '20260917'
host = 'https://orca.casa.ucl.ac.uk'
url = f'{host}/~jreades/data/{ymd}-{city}-listings.csv.gz'

#reproducible path read in

if Path.cwd().name == 'data_prep':
    dir_data = Path('../data')
else:
    dir_data = Path('data')
dir_raw = dir_data / 'raw'
dir_clean = dir_data / 'clean'

print(f'Data folder is: {dir_data}')
print(f'Raw data folder is: {dir_raw}')
print(f'Clean data folder is: {dir_clean}')

#since we are using multiple years each year needs a folder
#use the same file name as the url so it matches the qmd and doesn't download twice
path = dir_raw / ymd[:4] / Path(url).name
print(f'Writing out data to: {path}')


In [ ]:
#next we need to select the columns we need from our datasets

cols = ['id', 'listing_url', 'last_scraped', 'name',
    'description', 'host_id', 'host_name', 'host_since',
    'host_location', 'host_about', 'host_is_superhost',
    'host_listings_count', 'host_total_listings_count',
    'host_verifications', 'latitude', 'longitude',
    'property_type', 'room_type', 'accommodates',
    'bathrooms', 'bathrooms_text', 'bedrooms', 'beds',
    'amenities', 'price', 'minimum_nights', 'maximum_nights',
    'availability_365', 'number_of_reviews',
    'first_review', 'last_review', 'review_scores_rating',
    'license', 'reviews_per_month']

#extra columns that might be interesting for answering the qs
extra_cols = ['neighbourhood_cleansed',
    'number_of_reviews_ltm', 'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes', 'estimated_occupancy_l365d',
    'estimated_revenue_l365d']

cols = cols + extra_cols

print (len(cols))




In [15]:

#now we need to download the data and read it in.

def test_gzip(file_path):
    """Test if a file is a valid gzip file."""
    try:
        with gzip.open(file_path, 'rb') as f:
            f.read(1024 * 1024)
        return True
    except (OSError, EOFError):
        return False

def download_and_read_csvs(url, path):
    if not path.exists() or not test_gzip(path):
        if path.exists():
            path.unlink()  #remove file if it does exist but is not a valid gzip file
        print(f'Downloading data from {url}...')
        path.parent.mkdir(parents=True, exist_ok=True)  #make sure the directory exists
        with urlopen(url) as response, open(path, 'wb') as out_file:
            while chunk := response.read(1024 * 1024):
                out_file.write(chunk)

download_and_read_csvs(url, path)

df = pd.read_csv(path, compression = 'gzip', low_memory = False, usecols = cols, dtype = {'id' : str, 'host_id' : str})

print(f' dataframe is {df.shape[0]} rows by {df.shape[1]} columns')


 dataframe is 91954 rows by 40 columns


In [ ]:
#now we need to drop rows that have no id.

df = df.dropna(subset=['id'])
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

#now we check the columns
print(df.isnull().sum().sort_values(ascending = False)[:12])

empty_cols = df.columns[df.isnull().all().to_list()]
print(f'These columns are empty: {empty_cols}')
#also drop host_about, it's missing for about half the listings and would push normal rows over the cutoff
df = df.drop(columns=list(empty_cols) + ['host_about'])
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

#count missing values per row over the practical's columns only (not extra_cols)
probability = df.drop(columns=extra_cols).isnull().sum(axis = 1)

probability.plot.hist(bins = 100, figsize = (10, 5), title = 'Histogram of missing values per row')
print(probability.value_counts().sort_index())

#now we get rid of the rows that have more than 5 missing values from the practical.
cutoff = 5
dropped = df[probability > cutoff]
print(f'Dropping {(probability > cutoff).sum()} rows with more than {cutoff} missing values')

df = df[probability <= cutoff]
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

print(f"Dropped rows with any reviews: {(dropped.number_of_reviews > 0).sum()}")
